# Домашнее задание 4.2: Как провисает нить между опорами

Уравнение равновесия нити:
$$y'' = \frac{1}{a}\sqrt{1 + (y')^2}, \quad a = \frac{H}{q} > 0$$

---
## Часть 1. Опоры на одной высоте ($a = 1$, $y(0) = y(1) = 0$)

Решаем краевую задачу методом стрельбы (подбираем начальный наклон $s = y'(0)$ методом секущих) и сравниваем с точным решением:
$$y(x) = \cosh\left(x - \frac{1}{2}\right) - \cosh\left(\frac{1}{2}\right)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a_part1 = 1.0

def y_exact(x):
    return np.cosh(x - 0.5) - np.cosh(0.5)

# Интегрирование методом Рунге-Кутты 4 порядка
def solve_ode(s, h, a=1.0):
    n = int(round(1.0 / h))
    x = np.linspace(0.0, 1.0, n + 1)
    y = np.zeros(n + 1)
    dy = np.zeros(n + 1)
    dy[0] = s

    for i in range(n):
        k1_y = dy[i]
        k1_dy = (1.0 / a) * np.sqrt(1.0 + dy[i]**2)

        k2_y = dy[i] + 0.5 * h * k1_dy
        k2_dy = (1.0 / a) * np.sqrt(1.0 + k2_y**2)

        k3_y = dy[i] + 0.5 * h * k2_dy
        k3_dy = (1.0 / a) * np.sqrt(1.0 + k3_y**2)

        k4_y = dy[i] + h * k3_dy
        k4_dy = (1.0 / a) * np.sqrt(1.0 + k4_y**2)

        y[i+1] = y[i] + (h / 6.0) * (k1_y + 2*k2_y + 2*k3_y + k4_y)
        dy[i+1] = dy[i] + (h / 6.0) * (k1_dy + 2*k2_dy + 2*k3_dy + k4_dy)

    return x, y

# Метод стрельбы (поиск s методом секущих)
def shooting(h, s0=-1.0, s1=0.0, tol=1e-12):
    _, y0 = solve_ode(s0, h, a_part1)
    f0 = y0[-1]
    _, y1 = solve_ode(s1, h, a_part1)
    f1 = y1[-1]

    while abs(f1) > tol and abs(s1 - s0) > 1e-14:
        s_next = s1 - f1 * (s1 - s0) / (f1 - f0)
        s0, f0 = s1, f1
        s1 = s_next
        _, y1 = solve_ode(s1, h, a_part1)
        f1 = y1[-1]

    return s1

In [ ]:
# 1. Пристрелочные профили и сравнение с точным решением
h_base = 0.01
s_opt = shooting(h_base)
test_s = [-1.0, -0.7, -0.3, 0.0, s_opt]

plt.figure(figsize=(7, 5))
for s in test_s:
    x, y = solve_ode(s, h_base)
    label = f"s = {s:.3f} (найденный)" if s == s_opt else f"s = {s}"
    plt.plot(x, y, label=label)

plt.scatter([0, 1], [0, 0], color='red', s=60, zorder=5, label='Опоры')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.title("Пристрелочные профили при разных s")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True)
plt.show()

# Сопоставление с точным решением
x, y_num = solve_ode(s_opt, h_base)
y_true = y_exact(x)

plt.figure(figsize=(7, 5))
plt.plot(x, y_true, 'k-', linewidth=2, label="Точное решение")
plt.plot(x, y_num, 'r--', linewidth=2, label="Метод стрельбы")
plt.scatter([0, 1], [0, 0], color='blue', s=60, zorder=5, label='Опоры')
plt.title("Форма нити: численное и точное решение")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 2. Проверка точности при уменьшении шага
steps = [0.1, 0.05, 0.02, 0.01, 0.005]
max_errors = []

print(f"{'Шаг h':>8} | {'Наклон s':>12} | {'Невязка |y(1)|':>15} | {'Макс. ошибка':>15}")
print("-" * 60)

plt.figure(figsize=(7, 5))
for h in steps:
    s_h = shooting(h)
    x_h, y_h = solve_ode(s_h, h)
    err_profile = np.abs(y_h - y_exact(x_h))

    res_right = abs(y_h[-1])
    max_err = np.max(err_profile)
    max_errors.append(max_err)

    print(f"{h:8.4f} | {s_h:12.8f} | {res_right:15.3e} | {max_err:15.3e}")
    plt.plot(x_h, err_profile, label=f"h = {h}")

plt.yscale('log')
plt.title("Ошибка профиля при разных шагах интегрирования")
plt.xlabel("x")
plt.ylabel("|y_num - y_exact|")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
plt.loglog(steps, max_errors, 'bo-', label="Макс. ошибка профиля")
plt.title("Изменение ошибки при уменьшении шага h")
plt.xlabel("Шаг h")
plt.ylabel("Максимальная ошибка")
plt.legend()
plt.grid(True)
plt.show()

---
## Часть 2. Опоры на разных высотах и фиксированная длина нити

В этой части правая опора поднята на высоту $y(1) = 0.2$ (левая остаётся в $y(0) = 0$), а полная длина нити задана: $L = 1.2$.  
Теперь нам неизвестны сразу два параметра:
1. Начальный наклон на левой опоре $s = y'(0)$.
2. Параметр $a = H / q > 0$.

### Шаг 1. Расширенная система ОДУ с вычислением длины нити
Чтобы не считать интеграл длины отдельно после каждого шага, добавим в систему третье уравнение для текущей длины дуги $\ell(x)$:
$$\begin{cases}
y' = v \\
v' = \frac{1}{a}\sqrt{1 + v^2} \\
\ell' = \sqrt{1 + v^2}
\end{cases}$$
с начальными условиями в точке $x = 0$: $y(0) = 0$, $v(0) = s$, $\ell(0) = 0$.  
Решаем эту систему тем же методом Рунге-Кутты 4-го порядка.

In [ ]:
# Заданные параметры задачи
y_target = 0.2
L_target = 1.2

# Решение системы из 3 уравнений: для высоты y, производной dy и длины дуги l
def solve_ode_with_length(s, a, h):
    n = int(round(1.0 / h))
    x = np.linspace(0.0, 1.0, n + 1)
    y = np.zeros(n + 1)
    dy = np.zeros(n + 1)
    l = np.zeros(n + 1)
    
    # Начальные условия при x = 0
    dy[0] = s

    for i in range(n):
        # Коэффициенты k1
        dl1 = np.sqrt(1.0 + dy[i]**2)
        k1_y = dy[i]
        k1_dy = dl1 / a
        k1_l = dl1

        # Коэффициенты k2
        dy2 = dy[i] + 0.5 * h * k1_dy
        dl2 = np.sqrt(1.0 + dy2**2)
        k2_y = dy2
        k2_dy = dl2 / a
        k2_l = dl2

        # Коэффициенты k3
        dy3 = dy[i] + 0.5 * h * k2_dy
        dl3 = np.sqrt(1.0 + dy3**2)
        k3_y = dy3
        k3_dy = dl3 / a
        k3_l = dl3

        # Коэффициенты k4
        dy4 = dy[i] + h * k3_dy
        dl4 = np.sqrt(1.0 + dy4**2)
        k4_y = dy4
        k4_dy = dl4 / a
        k4_l = dl4

        # Шаг метода РК4
        y[i+1] = y[i] + (h / 6.0) * (k1_y + 2*k2_y + 2*k3_y + k4_y)
        dy[i+1] = dy[i] + (h / 6.0) * (k1_dy + 2*k2_dy + 2*k3_dy + k4_dy)
        l[i+1] = l[i] + (h / 6.0) * (k1_l + 2*k2_l + 2*k3_l + k4_l)

    return x, y, l

### Шаг 2. Вектор невязки и двумерный метод Ньютона

По условию параметр $a$ должен быть строго положительным ($a > 0$). Чтобы на итерациях Ньютона мы случайно не ушли в $a \le 0$, будем искать не сам параметр $a$, а его логарифм:
$$p = \ln a \quad \Longrightarrow \quad a = e^p > 0$$

Тогда вектор неизвестных равен $\mathbf{u} = (s, p)^T$, а вектор невязки в точке $x = 1$ состоит из двух условий (ошибка по высоте правого конца и ошибка по длине нити):
$$\mathbf{F}(s, p) = \begin{pmatrix} y(1; s, e^p) - 0.2 \\ \ell(1; s, e^p) - 1.2 \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \end{pmatrix}$$

Матрицу Якоби $J$ размера $2 \times 2$ считаем конечными разностями (сдвигаем по очереди $s$ и $p$ на малый шаг $\varepsilon = 10^{-6}$), а поправку $\Delta \mathbf{u}$ на каждом шаге находим из линейной системы $J \, \Delta \mathbf{u} = -\mathbf{F}$.

In [ ]:
# Функция вычисления вектора невязки F(u), где u = [s, log(a)]
def get_residual(u, h):
    s, log_a = u
    a = np.exp(log_a)
    _, y, l = solve_ode_with_length(s, a, h)
    return np.array([y[-1] - y_target, l[-1] - L_target])

# Численный расчёт матрицы Якоби 2x2 конечными разностями
def get_jacobian(u, F_current, h, eps=1e-6):
    J = np.zeros((2, 2))
    for j in range(2):
        u_shifted = u.copy()
        u_shifted[j] += eps
        F_shifted = get_residual(u_shifted, h)
        J[:, j] = (F_shifted - F_current) / eps
    return J

# Запуск двумерного метода Ньютона
h_newton = 0.01
# Начальное приближение: s = -0.5, a = 0.8 (log(a) = ln(0.8))
u = np.array([-0.5, np.log(0.8)])

history_u = []
history_res = []

print(f"{'Итер.':>5} | {'s':>12} | {'a':>12} | {'Невязка y(1)-0.2':>17} | {'Невязка l(1)-1.2':>17}")
print("-" * 75)

for it in range(20):
    F = get_residual(u, h_newton)
    history_u.append(u.copy())
    history_res.append(F.copy())
    
    s_cur, a_cur = u[0], np.exp(u[1])
    print(f"{it:5d} | {s_cur:12.8f} | {a_cur:12.8f} | {F[0]:17.3e} | {F[1]:17.3e}")
    
    # Проверка сходимости
    if np.linalg.norm(F) < 1e-12:
        break
        
    J = get_jacobian(u, F, h_newton)
    delta_u = np.linalg.solve(J, -F)
    u = u + delta_u

s_found = u[0]
a_found = np.exp(u[1])
print("\nНайденные параметры:")
print(f"Начальный наклон s = {s_found:.8f}")
print(f"Параметр a         = {a_found:.8f}")

### Шаг 3. Графики итераций Ньютона и убывания невязки

Построим, как менялась форма нити от начального приближения к точному профилю, а также как падали обе компоненты невязки с каждой итерацией.

In [ ]:
# 1. Изменение формы нити по итерациям Ньютона
plt.figure(figsize=(8, 5))

for idx, u_iter in enumerate(history_u):
    s_i = u_iter[0]
    a_i = np.exp(u_iter[1])
    x_i, y_i, _ = solve_ode_with_length(s_i, a_i, h_newton)
    
    # Чтобы не перегружать график, подпишем первые итерации и финальную
    if idx < 4 or idx == len(history_u) - 1:
        style = 'k-' if idx == len(history_u) - 1 else '--'
        lw = 2.2 if idx == len(history_u) - 1 else 1.2
        label = f"Итерация {idx} (финал)" if idx == len(history_u) - 1 else f"Итерация {idx}"
        plt.plot(x_i, y_i, style, linewidth=lw, label=label)

plt.scatter([0, 1], [0, y_target], color='red', s=65, zorder=5, label='Опоры (0, 0) и (1, 0.2)')
plt.title("Изменение формы нити по итерациям метода Ньютона")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True)
plt.show()

# 2. Графики обеих компонент невязки от номера итерации
res_array = np.array(history_res)
iters = np.arange(len(res_array))

plt.figure(figsize=(7, 5))
plt.semilogy(iters, np.abs(res_array[:, 0]), 'ro-', label="|y(1) - 0.2| (невязка высоты)")
plt.semilogy(iters, np.abs(res_array[:, 1]), 'bs--', label="|l(1) - 1.2| (невязка длины)")
plt.title("Убывание компонент невязки по итерациям Ньютона")
plt.xlabel("Номер итерации")
plt.ylabel("Модуль невязки (лог. шкала)")
plt.legend()
plt.grid(True)
plt.show()

### Шаг 4. Проверочный расчёт с более мелким шагом

Проверим найденные параметры $s$ и $a$ на более густой сетке ($h = 0.0005$, то есть в 20 раз мельче, чем при подборе), чтобы убедиться, что правый конец действительно попадает в $y(1) = 0.2$, а длина нити равна $L = 1.2$ с высокой точностью.

In [ ]:
# Проверочный расчёт с малым шагом h_check = 0.0005
h_check = 0.0005
x_chk, y_chk, l_chk = solve_ode_with_length(s_found, a_found, h_check)

err_y_check = abs(y_chk[-1] - y_target)
err_L_check = abs(l_chk[-1] - L_target)

print("=== Результаты проверочного расчёта (h = 0.0005) ===")
print(f"Найденный начальный наклон s : {s_found:.8f}")
print(f"Найденный параметр a         : {a_found:.8f}")
print(f"Значение y(1) на мелкой сетке: {y_chk[-1]:.10f} (цель: {y_target})")
print(f"Значение l(1) на мелкой сетке: {l_chk[-1]:.10f} (цель: {L_target})")
print(f"Ошибка высоты правого конца  : {err_y_check:.3e}")
print(f"Ошибка длины нити            : {err_L_check:.3e}")

### Ответ на вопрос: как длина нити определяет её провисание и натяжение?

1. **Влияние на провисание:**  
   Минимально возможная длина нити равна расстоянию по прямой между опорами: $L_{\min} = \sqrt{1^2 + 0.2^2} \approx 1.0198$.  
   - Чем **больше** длина нити $L$, тем сильнее она провисает вниз (точка минимума опускается ниже, а начальный наклон $s = y'(0)$ на левой опоре становится более крутым/отрицательным).  
   - При приближении $L$ к $L_{\min}$ провисание стремится к нулю, и нить вытягивается почти в прямую линию.

2. **Влияние на натяжение:**  
   Поскольку $a = H / q$, горизонтальная составляющая силы натяжения равна $H = a \cdot q$ (причём полное натяжение в любой точке нити $T(x) = H \sqrt{1 + (y')^2}$).  
   - При **увеличении** длины нити $L$ параметр $a$ уменьшается, а значит, **горизонтальное натяжение $H$ падает** (слабо натянутая длинная нить).  
   - При **уменьшении** длины нити $L \to L_{\min}$ параметр $a$ быстро растёт, и **натяжение нити $H$ стремится к бесконечности** (чтобы удерживать нить почти без провисания, её нужно тянуть с огромной силой).